In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import os
import datetime as dt

## Splitting genetic data into groups by genetically inferred ancestry

In [ ]:
# Make ancestry groups first to ensure that PRIMUS doesn't remove 
# individuals disproportionately from ancestry groups 

# Load in individual data from AoU general bucket(? is that what it's called)
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv . 

In [2]:
ancestry_df = pd.read_csv("ancestry_preds.tsv", sep="\t").dropna()
ancestry_df.columns = ["FID", "ancestry_pred", "probabilities", "pca_features", "ancestry_pred_other"]

In [ ]:
print((ancestry_df["ancestry_pred"].unique())) # eur afr amr sas eas mid

In [ ]:
# filter out individuals where ancestry_pred =/= ancestry_pred_other are filtered out
filt_df = ancestry_df[ancestry_df["ancestry_pred"] == ancestry_df["ancestry_pred_other"]]
filt_df = filt_df.drop("ancestry_pred_other", axis=1)

print("Filtered df contains", round(len(filt_df)/len(ancestry_df)*100, 2), "% of non-filtered df") # 93.86 %

filt_eur_df = filt_df[filt_df["ancestry_pred"] == "eur"]
filt_afr_df = filt_df[filt_df["ancestry_pred"] == "afr"]
filt_amr_df = filt_df[filt_df["ancestry_pred"] == "amr"]

print(len(filt_eur_df), len(filt_afr_df), len(filt_amr_df))

In [ ]:
# combine sex, age, and genetic ancestry prediction data into one dataframe

sex_df = pd.read_csv("v8_wgs_sex.psam", sep="\t").dropna()
sex_df.columns = ["FID", "person_id", "sex"]
sex_df["sex"] = sex_df["sex"].astype(np.int32)
age_df = pd.read_csv("v8_eff_ages_2024", sep="\t")
age_df["yob"] = 2024 - age_df["eff_age_2024"]
sexage_df = age_df.merge(sex_df[["FID", "person_id", "sex"]])

person_df = sexage_df.merge(filt_df[["FID", "ancestry_pred"]])
person_df

In [ ]:
kin_df = pd.read_csv("v8_arrays_maf01_geno05_deg4_subset.kin0", sep="\t").dropna()
print(min(kin_df["Kinship"])) # smallest kinship coefficient betwen any two individuals (0.0055)

In [ ]:
# Separating based on genetically-inferred ancestry: 'eur', 'afr', and 'amr' (European-like, African-like, American Admixed/Latino-like)

eur_df = person_df[person_df["ancestry_pred"] == "eur"]
afr_df = person_df[person_df["ancestry_pred"] == "afr"]
amr_df = person_df[person_df["ancestry_pred"] == "amr"]

print(len(eur_df), len(afr_df), len(amr_df)) # number of entries for each ancestry before kinship filtering

In [8]:
print('eur:', len(person_df[person_df["ancestry_pred"] == "eur"]))
print('afr:', len(person_df[person_df["ancestry_pred"] == "afr"]))
print('amr:', len(person_df[person_df["ancestry_pred"] == "amr"]))
print('sas:', len(person_df[person_df["ancestry_pred"] == "sas"]))
print('eas:', len(person_df[person_df["ancestry_pred"] == "eas"]))
print('mid:', len(person_df[person_df["ancestry_pred"] == "mid"]))

eur: 234319
afr: 84143
amr: 79104
sas: 5579
eas: 10099
mid: 1545


## Kinship filtering

In [ ]:
# read in KING data

king = pd.read_csv("v8_arrays_maf01_geno05_deg4_subset.kin0", sep="\t")
king.insert(2, "SID1", king["ID1"])
king.insert(5, "SID2", king["ID2"])

king

In [ ]:
king["InfType"].value_counts() # number of inidividuals with 4th / 3rd / 2nd / PO / FS / Dup/MZ inferred relationships

In [ ]:
# matching with ancestry-separated datasets

king_eur = king[king.ID1.isin(eur_df.person_id) & king.ID2.isin(eur_df.person_id)]
print(king_eur['InfType'].value_counts(),'\n')

king_afr = king[king.ID1.isin(afr_df.person_id) & king.ID2.isin(afr_df.person_id)]
print(king_afr['InfType'].value_counts(),'\n') 

king_amr = king[king.ID1.isin(amr_df.person_id) & king.ID2.isin(amr_df.person_id)]
print(king_amr['InfType'].value_counts()) 

In [ ]:
# example for AFR 
# view scatter of IBS0 (identity by state) x predicted kinship

sns.scatterplot(data=king_afr, x="Kinship", y="IBS0", hue="InfType");

In [ ]:
# add kinship values to per-ancestry DFs

ids_afr = pd.concat([king_afr['ID1'], king_afr['ID2'], afr_df['person_id']]).drop_duplicates().sort_values()
ids_afr = pd.DataFrame(ids_afr, columns=['person_id'])
ids_afr = ids_afr.merge(person_df, on='person_id')
ids_afr = ids_afr.set_index('person_id')

ids_amr = pd.concat([king_amr['ID1'], king_amr['ID2'], amr_df['person_id']]).drop_duplicates().sort_values()
ids_amr = pd.DataFrame(ids_amr, columns=['person_id'])
ids_amr = ids_amr.merge(person_df, on='person_id')
ids_amr = ids_amr.set_index('person_id')

ids_eur = pd.concat([king_eur['ID1'], king_eur['ID2'], eur_df['person_id']]).drop_duplicates().sort_values()
ids_eur = pd.DataFrame(ids_eur, columns=['person_id'])
ids_eur = ids_eur.merge(person_df, on='person_id')
ids_eur = ids_eur.set_index('person_id')

### Defining groups based on (family) relationships

In [ ]:
ids_afr['REL'] = [[] for _ in range(len(ids_afr))]
for i,row in king_afr.iterrows():
    ids_afr.loc[row['ID1'],'REL'].append((row['ID2'], row['InfType'], row['Kinship']))
    ids_afr.loc[row['ID2'],'REL'].append((row['ID1'], row['InfType'], row['Kinship']))
    
ids_amr['REL'] = [[] for _ in range(len(ids_amr))]
for i,row in king_amr.iterrows():
    ids_amr.loc[row['ID1'],'REL'].append((row['ID2'], row['InfType'], row['Kinship']))
    ids_amr.loc[row['ID2'],'REL'].append((row['ID1'], row['InfType'], row['Kinship']))

ids_eur['REL'] = [[] for _ in range(len(ids_eur))]
for i,row in king_eur.iterrows():
    ids_eur.loc[row['ID1'],'REL'].append((row['ID2'], row['InfType'], row['Kinship']))
    ids_eur.loc[row['ID2'],'REL'].append((row['ID1'], row['InfType'], row['Kinship']))
    
ids_amr

In [ ]:
# This defines edges as degrees of relationships and nodes as individuals
# rel_grps_[...] = list of lists = families and individuals belonging to these families

# AFR
edges = []
nodes = []
for i,row in ids_afr.iterrows():
    for j in row['REL']:
        if ids_afr.loc[i,'yob'] < ids_afr.loc[j[0],'yob']:
            edges.append((i, j[0], {'InfType': j[1], 'kinship': j[2]}))
        else:
            edges.append((j[0], i, {'InfType': j[1], 'kinship': j[2]}))
    ic = 'red' if row['sex'] == 1 else 'green'
    ia = row['yob']
    imf = row['sex']
    nodes.append((i, {'color': ic, 'yob': ia, 'sex': imf}))
G_afr = nx.Graph()
G_afr.add_nodes_from(nodes)
G_afr.add_edges_from(edges)
rel_grps_afr = [G_afr.subgraph(c) for c in nx.connected_components(G_afr)]
print(len(rel_grps_afr)) # how many families in AFR set


# AMR
edges = []
nodes = []
for i,row in ids_amr.iterrows():
    for j in row['REL']:
        if ids_amr.loc[i,'yob'] < ids_amr.loc[j[0],'yob']:
            edges.append((i, j[0], {'InfType': j[1], 'kinship': j[2]}))
        else:
            edges.append((j[0], i, {'InfType': j[1], 'kinship': j[2]}))
    ic = 'red' if row['sex'] == 1 else 'green'
    ia = row['yob']
    imf = row['sex']
    nodes.append((i, {'color': ic, 'yob': ia, 'sex': imf}))
G_amr = nx.Graph()
G_amr.add_nodes_from(nodes)
G_amr.add_edges_from(edges)
rel_grps_amr = [G_amr.subgraph(c) for c in nx.connected_components(G_amr)]
print(len(rel_grps_amr))

# EUR
edges = []
nodes = []
for i,row in ids_eur.iterrows():
    for j in row['REL']:
        if ids_eur.loc[i,'yob'] < ids_eur.loc[j[0],'yob']:
            edges.append((i, j[0], {'InfType': j[1], 'kinship': j[2]}))
        else:
            edges.append((j[0], i, {'InfType': j[1], 'kinship': j[2]}))
    ic = 'red' if row['sex'] == 1 else 'green'
    ia = row['yob']
    imf = row['sex']
    nodes.append((i, {'color': ic, 'yob': ia, 'sex': imf}))
G_eur = nx.Graph()
G_eur.add_nodes_from(nodes)
G_eur.add_edges_from(edges)
rel_grps_eur = [G_eur.subgraph(c) for c in nx.connected_components(G_eur)]

print(len(rel_grps_eur)) 

In [ ]:
# rel_grp_sizes_[..] = list of families (groups) and how many individuals are in each
# Below counts how many groups there are of a certain size

rel_grp_sizes_afr = [grp.number_of_nodes() for grp in rel_grps_afr]
display(pd.Series(rel_grp_sizes_afr).value_counts().sort_index())

rel_grp_sizes_amr = [grp.number_of_nodes() for grp in rel_grps_amr]
display(pd.Series(rel_grp_sizes_amr).value_counts().sort_index())

rel_grp_sizes_eur = [grp.number_of_nodes() for grp in rel_grps_eur]
display(pd.Series(rel_grp_sizes_eur).value_counts().sort_index())

rel_grp_sizes_amr

In [ ]:
# To visualise families/groups

def plot_fam(fam):
    rng = np.random.default_rng()
    fam_ages = [2024 - fam.nodes[node]['yob'] for node in fam.nodes]
    y_pos = [(age-min(fam_ages)) / (max(fam_ages) - min(fam_ages)) for age in fam_ages]
    top = np.linspace(0, 1, sum(i > 0.7 for i in y_pos))
    mid = np.linspace(0, 1, sum((i <= 0.7 and i >= 0.3) for i in y_pos))
    bot = np.linspace(0, 1, sum(i < 0.3 for i in y_pos))
    top = top - np.mean(top)
    mid = mid - np.mean(mid)
    bot = bot - np.mean(bot)
    i,j,k = 0,0,0
    x_pos = []
    for pos in y_pos:
        if pos > 0.7:
            x_pos.append(top[i])
            i += 1
        if pos <= 0.7 and pos >= 0.3:
            x_pos.append(mid[j])
            j += 1
        if pos < 0.3:
            x_pos.append(bot[k])
            k += 1
    pos = {node: (x_pos[i], y_pos[i]) for i, node in enumerate(fam.nodes)}
    #fig, ax = plt.subplots(figsize=(50,30))
    nx.draw(
        fam,
        with_labels=True,
        labels={node: f"\n{node}\nBorn {fam.nodes[node]['yob']}\n" for node in fam.nodes},
        node_color=[fam.nodes[node]['color'] for node in fam.nodes],
        pos=pos,
        horizontalalignment='center',
        verticalalignment='bottom',
        clip_on=False,
        #ax=ax
           )
    nx.draw_networkx_edge_labels(fam, pos=pos, edge_labels={edge: f"{fam.edges[edge]['InfType']}" for edge in fam.edges})

### Maximum independent set of unrelated individuals 
*(up to and including fourth degree)*

In [ ]:
# The command must be run per family group
# The result gives a list of individuals forming the maximum number per family 
# Code iterates over families per ancestry

# Note: nx.maximal_independent_set() does not solve the maximum independent set problem.

mis_afr = [nx.maximal_independent_set(fam) for fam in rel_grps_afr]
mis_amr = [nx.maximal_independent_set(fam) for fam in rel_grps_amr]
mis_eur = [nx.maximal_independent_set(fam) for fam in rel_grps_eur]

In [ ]:
print("Algorithm kept", len(mis_afr), "out of", len(ids_afr), 
      "AFR individuals (", round(len(mis_afr)/len(ids_afr)*100,2), "%)")

print("Algorithm kept", len(mis_amr), "out of", len(ids_amr), 
      "AMR individuals (", round(len(mis_amr)/len(ids_amr)*100,2), "%)")

print("Algorithm kept", len(mis_eur), "out of", len(ids_eur), 
      "EUR individuals (", round(len(mis_eur)/len(ids_eur)*100,2), "%)")

In [ ]:
# save lists of all IDs to keep

keep_afr = [x for l in mis_afr for x in l]
keep_amr = [x for l in mis_amr for x in l]
keep_eur = [x for l in mis_eur for x in l] 

In [ ]:
# Check

ids_afr = ids_afr.reset_index()
ids_amr = ids_amr.reset_index()
ids_eur = ids_eur.reset_index()

new_ids_afr = ids_afr[ids_afr['person_id'].isin(keep_afr)]
new_ids_amr = ids_amr[ids_amr['person_id'].isin(keep_amr)]
new_ids_eur = ids_eur[ids_eur['person_id'].isin(keep_eur)]

new_ids_afr['person_id'].to_csv('keep_unrelated_afr.txt', sep=" ", header = False, index = False)
new_ids_amr['person_id'].to_csv('keep_unrelated_amr.txt', sep=" ", header = False, index = False)
new_ids_eur['person_id'].to_csv('keep_unrelated_eur.txt', sep=" ", header = False, index = False)

In [ ]:
new_ids_afr[new_ids_afr['REL'].isin(['1st','2nd','3rd','PO', 'FR' ,'Dup/MZ'])] # ok

,FID,person_id,eff_age_2024,yob,sex,ancestry_pred,REL


In [ ]:
new_ids_amr[new_ids_amr['REL'].isin(['1st','2nd','3rd','PO', 'FR' ,'Dup/MZ'])] # ok

,FID,person_id,eff_age_2024,yob,sex,ancestry_pred,REL


In [ ]:
new_ids_eur[new_ids_eur['REL'].isin(['1st','2nd','3rd','PO', 'FR' ,'Dup/MZ'])] # should be empty >> ok

,FID,person_id,eff_age_2024,yob,sex,ancestry_pred,REL


In [ ]:
# Make index file to use in filtering array data

filt_afr = new_ids_afr[['FID', 'person_id']]
filt_afr.to_csv('keep_unrelated_afr_fidiid.txt', sep="\t", header = False, index = False)

filt_amr = new_ids_amr[['FID', 'person_id']]
filt_amr.to_csv('keep_unrelated_amr_fidiid.txt', sep="\t", header = False, index = False)

filt_eur = new_ids_eur[['FID', 'person_id']]
filt_eur.to_csv('keep_unrelated_eur_fidiid.txt', sep="\t", header = False, index = False)


# further cohort filtering will be done based on UMAP/PC plots
# these lists just contain the full list of unrelated individuals per ancestry group